In [0]:
library(tidyverse)
library(DBI)
library(dbplyr)
library(sparklyr)

In [0]:
sc <- sparklyr::spark_connect(method = "databricks")

In [0]:
dbGetQuery(
    conn = sc
    ,statement = "select current_user()"
)

In [0]:
dbExecute(
    conn = sc
    ,statement = "use mgiglia.hls_genie;"
)

In [0]:
%sql
select current_catalog(), current_schema();

In [0]:
patients_sdf <- sc |> 
    tbl(I("mgiglia.hls_genie.patients")) |>
    head(100)

encounters_sdf <- sc |> 
    tbl(I("mgiglia.hls_genie.encounters"))

patient_encounters <- patients_sdf |> 
    select(patient_id, birth_date, gender, ethnicity, marital, race, income) |> 
    inner_join(
        encounters_sdf |> select(encounter_id, patient_id, start, stop, base_encounter_cost, total_claim_cost, payer_coverage, encounter_class)
        ,by = join_by(patient_id == patient_id)) 

patient_encounters |> 
    count()

In [0]:
patient_encounters |> 
    mutate(patient_readmission_risk = sql(
        "patient_readmission_risk(
            birth_date
            ,gender
            ,ethnicity
            ,marital
            ,race
            ,income
            ,start
            ,stop
            ,base_encounter_cost
            ,total_claim_cost
            ,payer_coverage
            ,encounter_class
        )"
    )) |>
    sparklyr::spark_write_table("encounter_readmission_risk", options=list("overwrite = TRUE"))

In [0]:
readmit_risk <- tbl("encounter_readmission_risk")

In [0]:
readmit_risk_sdf <- tbl("readmit_risk")